# Baseline TF-IDF and LSA

## Goal
Build strong classical baselines before training learned embeddings.

## Required outputs
- TF-IDF feature stats
- LSA feature stats
- sentiment classification metrics
- category classification metrics
- first error examples

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.features.tfidf_features import build_vectorizer
from src.features.lsa_features import build_lsa
from src.models.linear_probe import build_logistic_probe

## 1. Data Loading
We will train on the training set and evaluate on the validation set. We keep the test set hidden until the final evaluation.

In [2]:
data_dir = PROJECT_ROOT / "data" / "processed"

# Load datasets
print("Loading datasets...")
train_df = pd.read_parquet(data_dir / "train.parquet")
val_df = pd.read_parquet(data_dir / "val.parquet")

# We will use 'cleaned_text', drop rows with missing text
train_df = train_df.dropna(subset=['cleaned_text', 'sentiment', 'main_category'])
val_df = val_df.dropna(subset=['cleaned_text', 'sentiment', 'main_category'])

# Subsample to avoid memory errors with SVD and linear probe
train_df = train_df.sample(n=250000, random_state=42)
val_df = val_df.sample(n=50000, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")

Loading datasets...


Train size: 250000
Validation size: 50000


## 2. Feature Extraction
We generate the TF-IDF and LSA representations.

In [3]:
# Initialize models
tfidf = build_vectorizer(max_features=25000, ngram_range=(1, 2))
lsa = build_lsa(n_components=100) # using 100 to save memory and time

print("Fitting TF-IDF...")
X_train_tfidf = tfidf.fit_transform(train_df['cleaned_text'])
X_val_tfidf = tfidf.transform(val_df['cleaned_text'])

print("Fitting LSA...")
X_train_lsa = lsa.fit_transform(X_train_tfidf)
X_val_lsa = lsa.transform(X_val_tfidf)

print(f"TF-IDF shape: {X_train_tfidf.shape}")
print(f"LSA shape: {X_train_lsa.shape}")

# Save feature extractors
models_dir = PROJECT_ROOT / "artifacts" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(tfidf, models_dir / "tfidf_vectorizer.pkl")
joblib.dump(lsa, models_dir / "lsa_transformer.pkl")
print("Saved feature extractors.")

Fitting TF-IDF...


Fitting LSA...


TF-IDF shape: (250000, 25000)
LSA shape: (250000, 100)


Saved feature extractors.


## 3. Modeling: Sentiment Classification
Predicting negative, neutral, positive.

In [4]:
y_train_sent = train_df['sentiment']
y_val_sent = val_df['sentiment']

print("Training Sentiment - TF-IDF...")
clf_sent_tfidf = build_logistic_probe()
clf_sent_tfidf.fit(X_train_tfidf, y_train_sent)

print("Training Sentiment - LSA...")
clf_sent_lsa = build_logistic_probe()
clf_sent_lsa.fit(X_train_lsa, y_train_sent)

print("--- Sentiment Classification Results ---")
y_pred_val_sent_tfidf = clf_sent_tfidf.predict(X_val_tfidf)
print(f"TF-IDF Validation Macro-F1: {f1_score(y_val_sent, y_pred_val_sent_tfidf, average='macro'):.4f}")

y_pred_val_sent_lsa = clf_sent_lsa.predict(X_val_lsa)
print(f"LSA Validation Macro-F1: {f1_score(y_val_sent, y_pred_val_sent_lsa, average='macro'):.4f}")

Training Sentiment - TF-IDF...


Training Sentiment - LSA...


--- Sentiment Classification Results ---


TF-IDF Validation Macro-F1: 0.6426


LSA Validation Macro-F1: 0.5288


## 4. Modeling: Category Classification
Predicting the 4 main categories.

In [5]:
y_train_cat = train_df['main_category']
y_val_cat = val_df['main_category']

print("Training Category - TF-IDF...")
clf_cat_tfidf = build_logistic_probe()
clf_cat_tfidf.fit(X_train_tfidf, y_train_cat)

print("Training Category - LSA...")
clf_cat_lsa = build_logistic_probe()
clf_cat_lsa.fit(X_train_lsa, y_train_cat)

print("--- Category Classification Results ---")
y_pred_val_cat_tfidf = clf_cat_tfidf.predict(X_val_tfidf)
print(f"TF-IDF Validation Macro-F1: {f1_score(y_val_cat, y_pred_val_cat_tfidf, average='macro'):.4f}")

y_pred_val_cat_lsa = clf_cat_lsa.predict(X_val_lsa)
print(f"LSA Validation Macro-F1: {f1_score(y_val_cat, y_pred_val_cat_lsa, average='macro'):.4f}")

Training Category - TF-IDF...


Training Category - LSA...


--- Category Classification Results ---


TF-IDF Validation Macro-F1: 0.7293


LSA Validation Macro-F1: 0.5168


## 5. Error Analysis (Category, LSA)

In [6]:
errors = val_df[y_val_cat != y_pred_val_cat_lsa].copy()
errors['predicted'] = y_pred_val_cat_lsa[y_val_cat != y_pred_val_cat_lsa]
errors[['text', 'main_category', 'predicted']].head(5)

,text,main_category,predicted
97762,Heavy and nice but they do let some light in,Home_and_Kitchen,Sports_and_Outdoors
146008,Yg,Beauty_and_Personal_Care,Sports_and_Outdoors
155252,Just what I was looking for. Flap stays off my...,Sports_and_Outdoors,Beauty_and_Personal_Care
194784,Great product will buy again 👌,Sports_and_Outdoors,Beauty_and_Personal_Care
31659,Not much I like but a lot I don't like. For on...,Electronics,Beauty_and_Personal_Care
